In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

Ver los nombres del objeto client (atributos y funciones)

In [10]:
import inspect
#print(dir(client))
print([name for name in dir(client) if not name.startswith("_")])

['callbacks', 'connections', 'get_prompt', 'get_resources', 'get_tools', 'session', 'tool_interceptors', 'tool_name_prefix']


In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [16]:
from pprint import pprint

print(response['messages'][-1].content)

Here’s a concise overview of the LangChain MCP Adapters library and what it’s for.

What it is
- langchain-mcp-adapters is a library that lets LangChain and LangGraph agents talk to tools that are exposed via the Model Context Protocol (MCP).
- It converts MCP tools into LangChain/LangGraph-compatible tools and lets you access tools from multiple MCP servers through a single client.
- It’s designed to reduce integration boilerplate when your agents need to use many external tools across one or more MCP servers.

Key concepts
- Model Context Protocol (MCP): An open protocol that standardizes how external tools and context are provided to LLMs. MCP servers publish tools that agents can invoke.
- MultiServerMCPClient: The primary client class for connecting to one or more MCP servers, loading MCP tools/prompts/resources, and exposing them as LangChain tools.
- Tool loading and translation: MCP tools (and prompts/resources) are loaded and converted into LangChain-compatible tools and promp

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [18]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [20]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response['messages'][-1].content)

('It’s 7:16 AM on Tuesday, July 7, 2026 in New York (Eastern Daylight Time, '
 'EDT). Want me to convert this to another time zone?')
